In [4]:
import pandas as pd
import numpy as np
import os

print("="*80)
print("STEP 3: DATA PREPROCESSING PIPELINE")
print("="*80)

# Load the training-ready data
df = pd.read_csv('data/credit_data_training_ready.csv')

print(f"\nDataset loaded successfully!")
print(f"  File: data/credit_data_training_ready.csv")
print(f"  Shape: {df.shape}")
print(f"  Rows: {len(df):,}")
print(f"  Columns: {df.shape[1]}")

# Quick check
print(f"\nColumns present:")
print(df.columns.tolist())

print(f"\nDefault rate: {df['default_flag'].mean()*100:.2f}%")

# Display sample
print("\nFirst 5 rows:")
df.head()

STEP 3: DATA PREPROCESSING PIPELINE

Dataset loaded successfully!
  File: data/credit_data_training_ready.csv
  Shape: (50000, 26)
  Rows: 50,000
  Columns: 26

Columns present:
['income_tier', 'employment_type', 'employment_tenure_months', 'education_level', 'residence_stability_years', 'dependents_count', 'upi_txn_frequency', 'avg_payment_delay_days', 'bill_payment_consistency', 'spending_volatility_cv', 'savings_ratio', 'utility_auto_payment_flag', 'bounce_rate_txn', 'wallet_min_balance', 'merchant_diversity_score', 'app_usage_regularity_days', 'device_stability_score', 'ecommerce_txn_frequency', 'recharge_regularity', 'social_media_linked_flag', 'kyc_completion_score', 'loan_inquiry_count_6m', 'existing_loan_flag', 'insurance_ownership_flag', 'account_age_months', 'default_flag']

Default rate: 22.00%

First 5 rows:


,income_tier,employment_type,employment_tenure_months,education_level,residence_stability_years,dependents_count,upi_txn_frequency,avg_payment_delay_days,bill_payment_consistency,spending_volatility_cv,...,device_stability_score,ecommerce_txn_frequency,recharge_regularity,social_media_linked_flag,kyc_completion_score,loan_inquiry_count_6m,existing_loan_flag,insurance_ownership_flag,account_age_months,default_flag
0,2,Gig_Worker,27,1,4,3,40,19.026109,0.841763,0.909225,...,0.905691,2,0.901327,0,1.0,0,0,0,15,0
1,5,Self_Employed,66,5,3,0,42,3.447501,0.773246,0.280324,...,0.700116,10,0.833506,1,1.0,0,1,1,81,0
2,3,Self_Employed,18,4,9,2,26,3.032418,0.832686,1.913300,...,0.629445,6,0.935097,1,0.5,1,0,1,33,0
3,3,Self_Employed,44,3,4,1,26,6.182045,0.966416,0.972937,...,0.607875,9,0.856647,0,1.0,0,1,0,47,0
4,2,Self_Employed,49,3,6,2,30,41.808930,0.674066,0.367610,...,0.842441,8,0.677063,0,1.0,3,0,1,57,0


In [5]:
print("="*80)
print("INITIAL DATA INSPECTION")
print("="*80)

# Check data types
print("\n1. Data Types:")
print("-"*80)
print(df.dtypes.value_counts())

# Check for missing values
print("\n2. Missing Values:")
print("-"*80)
missing = df.isnull().sum()
if missing.sum() == 0:
    print("  No missing values found!")
else:
    print(missing[missing > 0])

# Check for duplicates
print("\n3. Duplicate Rows:")
print("-"*80)
duplicates = df.duplicated().sum()
print(f"  Total duplicates: {duplicates}")

# Check target distribution
print("\n4. Target Variable Distribution:")
print("-"*80)
print(df['default_flag'].value_counts())
print(f"\nDefault rate: {df['default_flag'].mean()*100:.2f}%")

# Identify feature types
print("\n5. Feature Types:")
print("-"*80)
categorical_features = df.select_dtypes(include=['object']).columns.tolist()
numerical_features = df.select_dtypes(include=[np.number]).columns.tolist()
numerical_features.remove('default_flag')  # Remove target

print(f"  Categorical: {len(categorical_features)} features")
print(f"    {categorical_features}")
print(f"\n  Numerical: {len(numerical_features)} features")
print(f"    Total: {len(numerical_features)}")

INITIAL DATA INSPECTION

1. Data Types:
--------------------------------------------------------------------------------
int64      16
float64     9
object      1
Name: count, dtype: int64

2. Missing Values:
--------------------------------------------------------------------------------
  No missing values found!

3. Duplicate Rows:
--------------------------------------------------------------------------------
  Total duplicates: 0

4. Target Variable Distribution:
--------------------------------------------------------------------------------
default_flag
0    39000
1    11000
Name: count, dtype: int64

Default rate: 22.00%

5. Feature Types:
--------------------------------------------------------------------------------
  Categorical: 1 features
    ['employment_type']

  Numerical: 24 features
    Total: 24


In [6]:
print("="*80)
print("SEPARATING FEATURES AND TARGET")
print("="*80)

# Separate features (X) and target (y)
X = df.drop('default_flag', axis=1)
y = df['default_flag']

print(f"\nFeatures (X):")
print(f"  Shape: {X.shape}")
print(f"  Columns: {X.shape[1]}")

print(f"\nTarget (y):")
print(f"  Shape: {y.shape}")
print(f"  Unique values: {y.unique()}")
print(f"  Value counts:")
print(y.value_counts())

# Store feature names for later use
feature_names_original = X.columns.tolist()
print(f"\nOriginal feature names stored: {len(feature_names_original)} features")

SEPARATING FEATURES AND TARGET

Features (X):
  Shape: (50000, 25)
  Columns: 25

Target (y):
  Shape: (50000,)
  Unique values: [0 1]
  Value counts:
default_flag
0    39000
1    11000
Name: count, dtype: int64

Original feature names stored: 25 features


In [7]:
print("="*80)
print("ENCODING CATEGORICAL VARIABLES")
print("="*80)

# Identify categorical column
categorical_col = 'employment_type'

print(f"\nCategorical column: {categorical_col}")
print(f"Unique values: {X[categorical_col].unique()}")
print(f"Value counts:")
print(X[categorical_col].value_counts())

# Initialize LabelEncoder
label_encoder = LabelEncoder()

# Encode employment_type
X['employment_type_encoded'] = label_encoder.fit_transform(X[categorical_col])

# Display encoding mapping
print(f"\nEncoding mapping:")
print("-"*80)
for i, label in enumerate(label_encoder.classes_):
    count = (X['employment_type_encoded'] == i).sum()
    print(f"  {label:<20} → {i}  (Count: {count:,})")

# Drop original categorical column
X = X.drop(categorical_col, axis=1)

print(f"\nAfter encoding:")
print(f"  Shape: {X.shape}")
print(f"  Categorical columns remaining: {X.select_dtypes(include=['object']).shape[1]}")

# Update feature names
feature_names = X.columns.tolist()
print(f"  Total features: {len(feature_names)}")

ENCODING CATEGORICAL VARIABLES

Categorical column: employment_type
Unique values: ['Gig_Worker' 'Self_Employed' 'Student' 'Salaried' 'Unemployed']
Value counts:
employment_type
Salaried         15864
Self_Employed    15770
Gig_Worker       13306
Unemployed        2584
Student           2476
Name: count, dtype: int64

Encoding mapping:
--------------------------------------------------------------------------------
  Gig_Worker           → 0  (Count: 13,306)
  Salaried             → 1  (Count: 15,864)
  Self_Employed        → 2  (Count: 15,770)
  Student              → 3  (Count: 2,476)
  Unemployed           → 4  (Count: 2,584)

After encoding:
  Shape: (50000, 25)
  Categorical columns remaining: 0
  Total features: 25


In [8]:
print("="*80)
print("TRAIN-TEST SPLIT")
print("="*80)

# Split the data (70% train, 30% test)
# Use stratify to maintain default rate in both sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

print(f"\nSplit ratio: 70% train, 30% test")
print(f"Stratified by: default_flag")

print(f"\nTraining Set:")
print(f"  X_train shape: {X_train.shape}")
print(f"  y_train shape: {y_train.shape}")
print(f"  Samples: {len(X_train):,}")
print(f"  Default rate: {y_train.mean()*100:.2f}%")

print(f"\nTest Set:")
print(f"  X_test shape: {X_test.shape}")
print(f"  y_test shape: {y_test.shape}")
print(f"  Samples: {len(X_test):,}")
print(f"  Default rate: {y_test.mean()*100:.2f}%")

# Verify stratification worked
print(f"\nStratification verification:")
print(f"  Original default rate:  {y.mean()*100:.2f}%")
print(f"  Train default rate:     {y_train.mean()*100:.2f}%")
print(f"  Test default rate:      {y_test.mean()*100:.2f}%")
print(f"  Difference:             {abs(y_train.mean() - y_test.mean())*100:.2f}%")

TRAIN-TEST SPLIT

Split ratio: 70% train, 30% test
Stratified by: default_flag

Training Set:
  X_train shape: (35000, 25)
  y_train shape: (35000,)
  Samples: 35,000
  Default rate: 22.00%

Test Set:
  X_test shape: (15000, 25)
  y_test shape: (15000,)
  Samples: 15,000
  Default rate: 22.00%

Stratification verification:
  Original default rate:  22.00%
  Train default rate:     22.00%
  Test default rate:      22.00%
  Difference:             0.00%


In [9]:
print("="*80)
print("FEATURE SCALING (STANDARDIZATION)")
print("="*80)

# Initialize StandardScaler
scaler = StandardScaler()

# Fit on training data and transform both train and test
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Convert back to DataFrame for readability
X_train_scaled = pd.DataFrame(
    X_train_scaled,
    columns=feature_names,
    index=X_train.index
)

X_test_scaled = pd.DataFrame(
    X_test_scaled,
    columns=feature_names,
    index=X_test.index
)

print(f"\nScaling method: StandardScaler (mean=0, std=1)")
print(f"  Fitted on training data only")

print(f"\nBefore scaling (sample from X_train):")
print(X_train.head(3))

print(f"\nAfter scaling (sample from X_train_scaled):")
print(X_train_scaled.head(3))

# Verify scaling
print(f"\nScaling verification (training data):")
print(f"  Mean of scaled features: {X_train_scaled.mean().mean():.6f} (should be ~0)")
print(f"  Std of scaled features:  {X_train_scaled.std().mean():.6f} (should be ~1)")

# Show example of one feature before/after
example_feature = 'avg_payment_delay_days'
print(f"\nExample feature: {example_feature}")
print(f"  Before scaling - Mean: {X_train[example_feature].mean():.2f}, Std: {X_train[example_feature].std():.2f}")
print(f"  After scaling  - Mean: {X_train_scaled[example_feature].mean():.2f}, Std: {X_train_scaled[example_feature].std():.2f}")

FEATURE SCALING (STANDARDIZATION)

Scaling method: StandardScaler (mean=0, std=1)
  Fitted on training data only

Before scaling (sample from X_train):
       income_tier  employment_tenure_months  education_level  \
22520            3                        29                4   
10395            3                         6                4   
26536            2                        47                2   

       residence_stability_years  dependents_count  upi_txn_frequency  \
22520                          7                 0                 24   
10395                          0                 3                 39   
26536                         11                 4                 25   

       avg_payment_delay_days  bill_payment_consistency  \
22520                0.360146                  1.000000   
10395                2.062829                  1.000000   
26536                8.305976                  0.764948   

       spending_volatility_cv  savings_ratio  ...  device

In [10]:
print("="*80)
print("PREPROCESSING VALIDATION")
print("="*80)

# 1. Check for missing values after preprocessing
print("\n1. Missing Values Check:")
print("-"*80)
train_missing = X_train_scaled.isnull().sum().sum()
test_missing = X_test_scaled.isnull().sum().sum()
print(f"  Training set: {train_missing} missing values")
print(f"  Test set:     {test_missing} missing values")

# 2. Check feature counts
print("\n2. Feature Count Validation:")
print("-"*80)
print(f"  Original features:        {len(feature_names_original)}")
print(f"  After encoding:           {len(feature_names)}")
print(f"  Training set features:    {X_train_scaled.shape[1]}")
print(f"  Test set features:        {X_test_scaled.shape[1]}")

if X_train_scaled.shape[1] == X_test_scaled.shape[1]:
    print("  Status: Train and test have same features")
else:
    print("  ERROR: Feature mismatch!")

# 3. Check data types
print("\n3. Data Type Validation:")
print("-"*80)
print(f"  All features numeric: {X_train_scaled.dtypes.apply(lambda x: x in [np.float64, np.int64]).all()}")

# 4. Check for infinite values
print("\n4. Infinite Values Check:")
print("-"*80)
train_inf = np.isinf(X_train_scaled).sum().sum()
test_inf = np.isinf(X_test_scaled).sum().sum()
print(f"  Training set: {train_inf} infinite values")
print(f"  Test set:     {test_inf} infinite values")

# 5. Summary
print("\n5. Preprocessing Summary:")
print("-"*80)
print(f"  Total samples:               {len(df):,}")
print(f"  Training samples:            {len(X_train_scaled):,} (70%)")
print(f"  Test samples:                {len(X_test_scaled):,} (30%)")
print(f"  Features per sample:         {X_train_scaled.shape[1]}")
print(f"  Categorical encoding:        Done (employment_type)")
print(f"  Numerical scaling:           Done (StandardScaler)")
print(f"  Missing values:              0")
print(f"  Ready for model training:    Yes")

PREPROCESSING VALIDATION

1. Missing Values Check:
--------------------------------------------------------------------------------
  Training set: 0 missing values
  Test set:     0 missing values

2. Feature Count Validation:
--------------------------------------------------------------------------------
  Original features:        25
  After encoding:           25
  Training set features:    25
  Test set features:        25
  Status: Train and test have same features

3. Data Type Validation:
--------------------------------------------------------------------------------
  All features numeric: True

4. Infinite Values Check:
--------------------------------------------------------------------------------
  Training set: 0 infinite values
  Test set:     0 infinite values

5. Preprocessing Summary:
--------------------------------------------------------------------------------
  Total samples:               50,000
  Training samples:            35,000 (70%)
  Test samples:      

In [11]:
# Create directory for preprocessed data
os.makedirs('data/preprocessed', exist_ok=True)

print("="*80)
print("SAVING PREPROCESSED DATA")
print("="*80)

# Save training data
X_train_scaled.to_csv('data/preprocessed/X_train.csv', index=False)
y_train.to_csv('data/preprocessed/y_train.csv', index=False)

# Save test data
X_test_scaled.to_csv('data/preprocessed/X_test.csv', index=False)
y_test.to_csv('data/preprocessed/y_test.csv', index=False)

# Save feature names
with open('data/preprocessed/feature_names.txt', 'w') as f:
    f.write("PREPROCESSED FEATURE NAMES\n")
    f.write("="*80 + "\n\n")
    for i, name in enumerate(feature_names, 1):
        f.write(f"{i:2d}. {name}\n")

# Save preprocessing objects for later use
import joblib

os.makedirs('models/preprocessors', exist_ok=True)
joblib.dump(scaler, 'models/preprocessors/scaler.pkl')
joblib.dump(label_encoder, 'models/preprocessors/label_encoder.pkl')

print("\nFiles saved:")
print("-"*80)
print("  1. data/preprocessed/X_train.csv")
print(f"     Size: {os.path.getsize('data/preprocessed/X_train.csv') / (1024**2):.2f} MB")
print(f"     Shape: {X_train_scaled.shape}")

print("\n  2. data/preprocessed/y_train.csv")
print(f"     Size: {os.path.getsize('data/preprocessed/y_train.csv') / (1024):.2f} KB")
print(f"     Shape: {y_train.shape}")

print("\n  3. data/preprocessed/X_test.csv")
print(f"     Size: {os.path.getsize('data/preprocessed/X_test.csv') / (1024**2):.2f} MB")
print(f"     Shape: {X_test_scaled.shape}")

print("\n  4. data/preprocessed/y_test.csv")
print(f"     Size: {os.path.getsize('data/preprocessed/y_test.csv') / (1024):.2f} KB")
print(f"     Shape: {y_test.shape}")

print("\n  5. data/preprocessed/feature_names.txt")
print("\n  6. models/preprocessors/scaler.pkl")
print("  7. models/preprocessors/label_encoder.pkl")

print("\n" + "="*80)
print("PREPROCESSING COMPLETE!")
print("="*80)

SAVING PREPROCESSED DATA

Files saved:
--------------------------------------------------------------------------------
  1. data/preprocessed/X_train.csv
     Size: 16.40 MB
     Shape: (35000, 25)

  2. data/preprocessed/y_train.csv
     Size: 102.55 KB
     Shape: (35000,)

  3. data/preprocessed/X_test.csv
     Size: 7.03 MB
     Shape: (15000, 25)

  4. data/preprocessed/y_test.csv
     Size: 43.96 KB
     Shape: (15000,)

  5. data/preprocessed/feature_names.txt

  6. models/preprocessors/scaler.pkl
  7. models/preprocessors/label_encoder.pkl

PREPROCESSING COMPLETE!


In [12]:
print("="*80)
print("STEP 3 COMPLETE - PREPROCESSING SUMMARY")
print("="*80)

print(f"""
PREPROCESSING SUMMARY
{'='*80}

Input Data:
  Original dataset:           50,000 rows × 26 columns
  Features:                   25
  Target variable:            default_flag
  
Preprocessing Steps:
  1. Categorical encoding:    employment_type → employment_type_encoded
  2. Feature scaling:         StandardScaler (mean=0, std=1)
  3. Train-test split:        70-30 stratified split
  
Output Data:
  Training set (X_train):     {len(X_train_scaled):,} rows × {X_train_scaled.shape[1]} columns
  Training labels (y_train):  {len(y_train):,} samples
  Test set (X_test):          {len(X_test_scaled):,} rows × {X_test_scaled.shape[1]} columns
  Test labels (y_test):       {len(y_test):,} samples
  
Data Quality:
  Missing values:             0
  Infinite values:            0
  Features normalized:        All {X_train_scaled.shape[1]} features
  Default rate preserved:     {y_train.mean()*100:.2f}% (train) vs {y_test.mean()*100:.2f}% (test)
  
Files Created:
  - data/preprocessed/X_train.csv
  - data/preprocessed/y_train.csv
  - data/preprocessed/X_test.csv
  - data/preprocessed/y_test.csv
  - data/preprocessed/feature_names.txt
  - models/preprocessors/scaler.pkl
  - models/preprocessors/label_encoder.pkl

Status: Ready for STEP 4 (Model Training)
""")

# Display sample of preprocessed data
print("\nSample Preprocessed Data (X_train_scaled):")
print("="*80)
X_train_scaled.head()

STEP 3 COMPLETE - PREPROCESSING SUMMARY

PREPROCESSING SUMMARY

Input Data:
  Original dataset:           50,000 rows × 26 columns
  Features:                   25
  Target variable:            default_flag
  
Preprocessing Steps:
  1. Categorical encoding:    employment_type → employment_type_encoded
  2. Feature scaling:         StandardScaler (mean=0, std=1)
  3. Train-test split:        70-30 stratified split
  
Output Data:
  Training set (X_train):     35,000 rows × 25 columns
  Training labels (y_train):  35,000 samples
  Test set (X_test):          15,000 rows × 25 columns
  Test labels (y_test):       15,000 samples
  
Data Quality:
  Missing values:             0
  Infinite values:            0
  Features normalized:        All 25 features
  Default rate preserved:     22.00% (train) vs 22.00% (test)
  
Files Created:
  - data/preprocessed/X_train.csv
  - data/preprocessed/y_train.csv
  - data/preprocessed/X_test.csv
  - data/preprocessed/y_test.csv
  - data/preprocessed/feat

,income_tier,employment_tenure_months,education_level,residence_stability_years,dependents_count,upi_txn_frequency,avg_payment_delay_days,bill_payment_consistency,spending_volatility_cv,savings_ratio,...,device_stability_score,ecommerce_txn_frequency,recharge_regularity,social_media_linked_flag,kyc_completion_score,loan_inquiry_count_6m,existing_loan_flag,insurance_ownership_flag,account_age_months,employment_type_encoded
22520,0.335872,-0.077844,1.089419,0.359266,-1.211039,-1.108766,-0.661113,0.758498,-0.708795,-0.459028,...,-1.178908,0.700330,0.290684,-1.389103,0.464017,0.691269,1.344568,-0.927426,-1.100449,-0.282015
10395,0.335872,-0.973553,1.089419,-1.281617,1.017946,0.341204,-0.553028,0.758498,-0.233532,0.490494,...,0.130161,-0.327379,-0.224897,0.719889,0.464017,-1.053227,1.344568,1.078253,-0.522533,0.649079
26536,-0.609637,0.623145,-0.559110,1.296914,1.760941,-1.012101,-0.156718,-0.535767,0.530039,-0.630577,...,-0.758871,-0.669949,0.054151,-1.389103,0.464017,0.691269,-0.743734,-0.927426,-0.955970,0.649079
18272,0.335872,2.570338,-0.559110,-0.578381,-0.468044,0.341204,-0.678699,0.128096,-0.614634,-0.682981,...,-0.912470,-1.355089,0.788956,0.719889,0.464017,2.435764,1.344568,-0.927426,-0.493638,-0.282015
49139,-0.609637,-0.739890,-0.559110,-1.281617,1.760941,0.244540,1.096402,-1.320146,-0.556829,-0.467601,...,1.312432,-1.012519,1.203645,0.719889,0.464017,-1.053227,-0.743734,1.078253,-0.551429,0.649079
